In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU chưa được kích hoạt trên Colab!")

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [2]:
# ==================================================================================
# CELL MOUNT DRIVE DÙNG CHUNG — giữ NGUYÊN VĂN ở mọi notebook experiment
# (Baseline, Label Flipping, Backdoor, Defense). Chỉ sửa khối CẤU HÌNH bên dưới,
# và phải sửa giống hệt nhau ở mọi notebook.
# ==================================================================================
import os
import sys
import json    # dùng ở các cell sau
import csv     # dùng ở các cell sau
import shutil
import subprocess
from pathlib import Path

# ------------------------------ CẤU HÌNH (cả nhóm dùng chung) ---------------------
SHARED_FOLDER_NAME = 'data-poisoning-shared'   # tên thư mục gốc trong Drive (mọi người đặt đúng tên này)
REPO_URL = 'https://github.com/qtrong0205/data-poisoning-research'                                  # URL GitHub của project; để trống = lấy code từ <thư mục gốc>/repo
REPO_BRANCH = 'main'
LOCAL_PROJECT = Path('/content/data-poisoning-plantvillage')   # nơi chạy code (đĩa local của Colab, nhanh)
DRIVE_MOUNT = '/content/drive'
PIPELINES = ['baseline', 'label_flipping', 'backdoor', 'defenses']
# -----------------------------------------------------------------------------------

# 1. Mount Drive -----------------------------------------------------------------------
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
elif not Path(DRIVE_MOUNT).is_dir():
    raise RuntimeError('Cell này dành cho Google Colab (hoặc đặt DRIVE_MOUNT trỏ tới thư mục Drive).')

# 2. Tìm thư mục gốc dùng chung ----------------------------------------------------------
_candidates = []
if os.environ.get('DP_DRIVE_ROOT_OVERRIDE'):          # chỉ dùng khi muốn ép đường dẫn thủ công
    _candidates.append(Path(os.environ['DP_DRIVE_ROOT_OVERRIDE']))
_candidates.append(Path(DRIVE_MOUNT) / 'MyDrive' / SHARED_FOLDER_NAME)
_candidates.append(Path(DRIVE_MOUNT) / 'Shareddrives' / SHARED_FOLDER_NAME)
_candidates.extend(sorted(Path(DRIVE_MOUNT, 'Shareddrives').glob(f'*/{SHARED_FOLDER_NAME}')))
DRIVE_ROOT = next((c for c in _candidates if c.is_dir()), None)
if DRIVE_ROOT is None:
    raise FileNotFoundError(
        f'Không tìm thấy thư mục dùng chung "{SHARED_FOLDER_NAME}". Đã tìm ở:\n  - '
        + '\n  - '.join(str(c) for c in _candidates)
        + '\nCách khắc phục: mở thư mục được chia sẻ trong Drive (Shared with me), chuột phải '
          '"Add shortcut to Drive", chọn My Drive và giữ nguyên tên, rồi chạy lại cell này.'
    )

DRIVE_DATA = DRIVE_ROOT / 'data'
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_CHECKPOINTS = DRIVE_ROOT / 'checkpoints'

# 3. Bảo đảm cấu trúc thư mục (chỉ tạo thư mục còn thiếu, không xóa/ghi đè gì) -----------------
for _d in [DRIVE_DATA / 'plantvillage dataset', DRIVE_DATA / 'splits', DRIVE_DATA / 'samples',
           DRIVE_DATA / 'poisoned' / 'label_flipping', DRIVE_DATA / 'poisoned' / 'backdoor',
           *(DRIVE_RESULTS / p for p in PIPELINES), *(DRIVE_CHECKPOINTS / p for p in PIPELINES)]:
    _d.mkdir(parents=True, exist_ok=True)

# Kiểm tra quyền ghi (người chỉ có quyền xem sẽ không lưu được kết quả)
_probe = DRIVE_RESULTS / f'.write_test_{os.getpid()}'
try:
    _probe.write_text('ok')
    _probe.unlink()
except OSError as exc:
    raise PermissionError(
        f'Không ghi được vào {DRIVE_RESULTS}. Hãy nhờ chủ thư mục cấp quyền Editor. ({exc})') from exc


# 4. Lấy code về đĩa local ------------------------------------------------------------------
def _sh(cmd, secret=None):
    shown = ' '.join(cmd)
    if secret:
        shown = shown.replace(secret, '***')
    print('$', shown)
    done = subprocess.run(cmd, text=True, capture_output=True)
    out = (done.stdout + done.stderr).strip()
    if secret:
        out = out.replace(secret, '***')
    if out:
        print(out)
    if done.returncode != 0:
        raise RuntimeError(f'Lệnh thất bại (exit {done.returncode}).')


if REPO_URL:
    _url, _token = REPO_URL, None
    if _url.startswith('https://') and 'github.com' in _url:
        try:                                    # repo private: lưu token trong Colab Secrets với tên GITHUB_TOKEN
            from google.colab import userdata
            _token = userdata.get('GITHUB_TOKEN')
        except Exception:
            _token = None
        if _token:
            _url = _url.replace('https://', f'https://{_token}@', 1)
    if (LOCAL_PROJECT / '.git').is_dir():
        _sh(['git', '-C', str(LOCAL_PROJECT), 'pull', '--ff-only'], secret=_token)
    else:
        _sh(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, _url, str(LOCAL_PROJECT)], secret=_token)
    CODE_SOURCE = f'GitHub {REPO_URL} ({REPO_BRANCH})'
else:
    _src = DRIVE_ROOT / 'repo'
    if not _src.is_dir():
        raise FileNotFoundError(
            f'Chưa có {_src}. Hãy upload code vào thư mục repo/ trong Drive dùng chung '
            '(chỉ code, không kèm data/results/checkpoints), hoặc điền REPO_URL.')
    _skip_anywhere = {'.git', '__pycache__', '.ipynb_checkpoints', 'results', 'checkpoints'}
    _skip_top = {'data', 'outputs', 'models'}

    def _ignore(directory, names):
        skip = {n for n in names if n in _skip_anywhere or n.endswith(('.pth', '.pt', '.onnx'))}
        if Path(directory) == _src:
            skip |= {n for n in names if n in _skip_top}
        return skip

    shutil.copytree(_src, LOCAL_PROJECT, dirs_exist_ok=True, ignore=_ignore)
    CODE_SOURCE = f'Drive: {_src}'

PROJECT_ROOT = LOCAL_PROJECT
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 5. Nối các thư mục dữ liệu/kết quả/checkpoint của project vào Drive (symlink) ---------------
#    Code không cần sửa: đường dẫn tương đối trong project tự trỏ vào Drive.
LINKS = {
    'data/plantvillage dataset': DRIVE_DATA / 'plantvillage dataset',
    'data/splits': DRIVE_DATA / 'splits',
    'data/poisoned': DRIVE_DATA / 'poisoned',
    'data/samples': DRIVE_DATA / 'samples',
    'outputs': DRIVE_RESULTS,
    'models/checkpoints': DRIVE_CHECKPOINTS,
}
for _p in PIPELINES:
    LINKS[f'experiments/{_p}/results'] = DRIVE_RESULTS / _p
    LINKS[f'experiments/{_p}/checkpoints'] = DRIVE_CHECKPOINTS / _p


def _link(rel, target):
    path = PROJECT_ROOT / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.is_symlink():
        if Path(os.readlink(path)) == target:
            return 'OK'
        path.unlink()
    elif path.exists():
        if path.is_dir() and not any(path.iterdir()):
            path.rmdir()
        else:
            return 'BỎ QUA: đã có dữ liệu thật trong thư mục local, không ghi đè'
    os.symlink(target, path, target_is_directory=True)
    return 'OK'


_link_status = {rel: _link(rel, tgt) for rel, tgt in LINKS.items()}

# 6. Biến môi trường cho code mới (attack/defense) ---------------------------------------------
os.environ.update({
    'DP_DRIVE_ROOT': str(DRIVE_ROOT),
    'DP_DATA_ROOT': str(DRIVE_DATA),
    'DP_RESULTS_ROOT': str(DRIVE_RESULTS),
    'DP_CHECKPOINTS_ROOT': str(DRIVE_CHECKPOINTS),
})


def finish_session():
    """Gọi ở cuối phiên (hoặc trước khi bạn khác tiếp tục) để chắc chắn dữ liệu đã đồng bộ lên Drive."""
    if IN_COLAB:
        from google.colab import drive as _drive
        _drive.flush_and_unmount()
        print('Đã đồng bộ và ngắt kết nối Drive.')
    else:
        print('Không ở trong Colab: không cần đồng bộ.')


# 7. Tóm tắt -----------------------------------------------------------------------------------
print(f'Drive root : {DRIVE_ROOT}')
print(f'Code       : {PROJECT_ROOT}  <-  {CODE_SOURCE}')
print('Liên kết (project -> Drive):')
for _rel, _st in _link_status.items():
    print(f'  {_rel:<40} {_st}')
_raw = DRIVE_DATA / 'plantvillage dataset' / 'color'
_has = {
    'ảnh gốc (data/plantvillage dataset/color)': _raw.is_dir() and any(_raw.iterdir()),
    'manifest_train.csv': (DRIVE_DATA / 'splits' / 'manifest_train.csv').is_file(),
    'manifest_test.csv': (DRIVE_DATA / 'splits' / 'manifest_test.csv').is_file(),
    'class_to_idx.json': (DRIVE_DATA / 'splits' / 'class_to_idx.json').is_file(),
}
print('Dữ liệu dùng chung:')
for _name, _ok in _has.items():
    print(f'  {"CÓ   " if _ok else "THIẾU"} {_name}')
print('Quyền ghi: OK')

Mounted at /content/drive
$ git clone --depth 1 --branch main https://github.com/qtrong0205/data-poisoning-research /content/data-poisoning-plantvillage
Cloning into '/content/data-poisoning-plantvillage'...
Drive root : /content/drive/MyDrive/data-poisoning-shared
Code       : /content/data-poisoning-plantvillage  <-  GitHub https://github.com/qtrong0205/data-poisoning-research (main)
Liên kết (project -> Drive):
  data/plantvillage dataset                OK
  data/splits                              OK
  data/poisoned                            OK
  data/samples                             OK
  outputs                                  OK
  models/checkpoints                       OK
  experiments/baseline/results             OK
  experiments/baseline/checkpoints         OK
  experiments/label_flipping/results       OK
  experiments/label_flipping/checkpoints   OK
  experiments/backdoor/results             OK
  experiments/backdoor/checkpoints         OK
  experiments/defenses/results

In [3]:
# Kiểm tra các thư mục kết quả/checkpoint đã thật sự nối vào Drive (nếu không, kết quả sẽ MẤT khi hết phiên)
_bad = {rel: st for rel, st in _link_status.items() if st != 'OK'}
if _bad:
    raise RuntimeError(
        'Các thư mục sau chưa được nối vào Drive:\n'
        + '\n'.join(f'  {rel}: {st}' for rel, st in _bad.items())
        + '\nNguyên nhân thường gặp: thư mục đó vẫn còn file được commit trong repo (hãy git rm --cached '
          'và thêm vào .gitignore). Không chạy training khi lỗi này chưa được xử lý.')
print(f'Đã nối {len(_link_status)} thư mục của project vào Drive.')

Đã nối 14 thư mục của project vào Drive.


In [4]:
# Cài dependency, bỏ qua torch/torchvision/CUDA để không ghi đè PyTorch sẵn có của Colab
import re

SKIP_PACKAGES = {'torch', 'torchvision', 'torchaudio', 'triton'}
packages = []
for line in (PROJECT_ROOT / 'requirements.txt').read_text(encoding='utf-8').splitlines():
    item = line.strip()
    if not item or item.startswith(('#', '-')):
        continue
    name = re.split(r'[=<>!~;\[\s@]', item, maxsplit=1)[0].lower().replace('_', '-')
    if name in SKIP_PACKAGES or name.startswith('nvidia-'):
        continue
    packages.append(item)
if packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print(f'Đã cài {len(packages)} package.')

Đã cài 13 package.


In [5]:
import torch
import torchvision
import yaml
import sklearn
import matplotlib

print("Dependencies imported successfully!")

Dependencies imported successfully!


In [6]:
# Kiểm tra dữ liệu dùng chung. Notebook KHÔNG tự tải dataset hay tạo split:
# việc đó do người quản lý làm một lần (nhiều người cùng ghi vào Drive dùng chung sẽ gây lỗi).
import pandas as pd

split_dir = PROJECT_ROOT / 'data/splits'
for _name in ('manifest_train.csv', 'manifest_test.csv', 'class_to_idx.json'):
    if not (split_dir / _name).is_file():
        raise FileNotFoundError(f'Thiếu {split_dir / _name}. Hãy báo người quản lý chuẩn bị dữ liệu.')

# So số ảnh trong manifest với số file thật của từng lớp (nhanh, chỉ liệt kê thư mục)
_problems = []
for _split in ('train', 'test'):
    _df = pd.read_csv(split_dir / f'manifest_{_split}.csv')
    for _cls, _n in _df.groupby('class_name').size().items():
        _folder = split_dir / _split / _cls
        _have = sum(1 for e in os.scandir(_folder) if e.is_file()) if _folder.is_dir() else 0
        if _have < _n:
            _problems.append({'split': _split, 'class': _cls, 'manifest': int(_n), 'on_drive': _have})
if _problems:
    display(pd.DataFrame(_problems))
    raise FileNotFoundError(f'{len(_problems)} lớp thiếu ảnh trong data/splits (bảng trên). Báo người quản lý bổ sung trước khi train.')
print('data/splits đầy đủ so với manifest.')

data/splits đầy đủ so với manifest.


In [7]:
# Hàm chạy lệnh con và in log RA CELL NGAY KHI có (kể cả thanh tiến trình tqdm dùng ký tự \r).
# subprocess.run(...) trần không hiện log trong cell Colab, nên cell trông như bị "đóng băng".
import codecs
import os
import subprocess
import sys


def run_streaming(cmd, cwd=None):
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    decoder = codecs.getincrementaldecoder('utf-8')('replace')   # không làm hỏng chữ có dấu bị cắt giữa chunk
    try:
        while True:
            chunk = proc.stdout.read1(4096)       # trả về ngay khi có dữ liệu, không chờ xuống dòng
            if not chunk:
                break
            sys.stdout.write(decoder.decode(chunk))
            sys.stdout.flush()
        sys.stdout.write(decoder.decode(b'', final=True))
    except BaseException:                         # bấm Stop trên Colab: dừng luôn tiến trình train, tránh chạy mồ côi
        proc.kill()
        proc.wait()
        raise
    code = proc.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)
    return code


# ---------------------------------------------------------------------------------
# Đọc hàng chục nghìn ảnh nhỏ trực tiếp từ Drive rất chậm và có thể đứng rất lâu (nhất là lần đọc đầu).
# Chép ảnh về đĩa local của Colab (song song) rồi train từ bản local. Kết quả/checkpoint vẫn ghi vào Drive.
# ---------------------------------------------------------------------------------
import shutil
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd

USE_LOCAL_SPLITS = True                       # False: đọc thẳng từ Drive (chậm)
LOCAL_SPLITS = Path('/content/splits_local')


def prepare_local_splits(only_subset, workers=32):
    if not USE_LOCAL_SPLITS:
        print('USE_LOCAL_SPLITS=False: đọc ảnh trực tiếp từ Drive.', flush=True)
        return
    src = DRIVE_DATA / 'splits'
    train = pd.read_csv(src / 'manifest_train.csv')
    if only_subset:
        train = train[train.in_subset == 1]
    test = pd.read_csv(src / 'manifest_test.csv')
    rels = list(train.rel_path) + list(test.rel_path)
    todo = [r for r in rels if not (LOCAL_SPLITS / r).is_file()]
    print(f'Chép {len(todo):,}/{len(rels):,} ảnh từ Drive về {LOCAL_SPLITS} (đã có sẵn {len(rels) - len(todo):,}).', flush=True)
    LOCAL_SPLITS.mkdir(parents=True, exist_ok=True)
    for name in ('manifest_train.csv', 'manifest_test.csv', 'class_to_idx.json'):
        shutil.copy2(src / name, LOCAL_SPLITS / name)

    def _copy(rel):
        dst = LOCAL_SPLITS / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        tmp = dst.with_name(dst.name + '.part')
        shutil.copyfile(src / rel, tmp)
        os.replace(tmp, dst)                  # ghi xong mới đổi tên: file dở dang không bị coi là đã chép

    t0, done = time.time(), 0
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for _ in pool.map(_copy, todo):
            done += 1
            if done % 2000 == 0:
                print(f'  đã chép {done:,}/{len(todo):,} ({done / (time.time() - t0):.0f} ảnh/giây)', flush=True)
    print(f'Xong {len(todo):,} ảnh trong {time.time() - t0:.0f}s.', flush=True)

    link = PROJECT_ROOT / 'data/splits'
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        raise RuntimeError(f'{link} là thư mục thật, không đổi sang bản local được.')
    link.symlink_to(LOCAL_SPLITS, target_is_directory=True)
    print(f'data/splits đang trỏ tới {LOCAL_SPLITS} (đĩa local).', flush=True)

In [9]:
# (Tùy chọn, vài giây) Đo tốc độ đọc ảnh từ Drive để biết 1 epoch sẽ mất khoảng bao lâu
import time

_train_df = pd.read_csv(split_dir / 'manifest_train.csv')
_sub = _train_df[_train_df.in_subset == 1]
_sample = _sub.sample(n=min(100, len(_sub)), random_state=1)
_t0 = time.time()
for _p in _sample.rel_path:
    with open(split_dir / _p, 'rb') as _f:
        _f.read()
_ms = (time.time() - _t0) / len(_sample) * 1000
_minutes = _ms * len(_sub) / 1000 / 60
_test_sample = pd.read_csv(split_dir / 'manifest_test.csv').sample(n=100, random_state=7)
_times = []
for _p in _test_sample.rel_path:
    _a = time.time()
    with open(split_dir / _p, 'rb') as _f:
        _f.read()
    _times.append(time.time() - _a)
print(f'Ảnh TEST (chưa đọc lần nào): trung bình {sum(_times) / len(_times) * 1000:.1f} ms/ảnh, chậm nhất {max(_times):.2f}s.')
print(f'Đọc ảnh từ Drive: {_ms:.1f} ms/ảnh.')
print(f'Ước tính đọc hết {len(_sub):,} ảnh Sweep bằng 1 luồng: {_minutes:.1f} phút (chia cho số num_workers khi train).')
if _ms > 30:
    print('Đọc từ Drive chậm: epoch đầu có thể rất lâu và im lặng. Cân nhắc chép data/splits về đĩa /content trước khi train.')

Ảnh TEST (chưa đọc lần nào): trung bình 106.4 ms/ảnh, chậm nhất 0.46s.
Đọc ảnh từ Drive: 1.3 ms/ảnh.
Ước tính đọc hết 13,028 ảnh Sweep bằng 1 luồng: 0.3 phút (chia cho số num_workers khi train).


In [10]:
%cd /content/data-poisoning-plantvillage


def _baseline_run_complete(_mode, _seed):
    _run_root = DRIVE_RESULTS / 'baseline' / _mode / f'seed_{_seed}'
    _metrics_path = _run_root / 'metrics.json'
    _checkpoint_path = DRIVE_CHECKPOINTS / 'baseline' / _mode / f'seed_{_seed}' / 'final_model.pth'
    print(f'Checking {_mode} seed {_seed}: metrics={_metrics_path.is_file()}, checkpoint={_checkpoint_path.is_file()}', flush=True)
    if not (_metrics_path.is_file() and _checkpoint_path.is_file()):
        print(f'{_mode} seed {_seed}: training is required.', flush=True)
        return False
    try:
        _metrics = json.loads(_metrics_path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError) as _exc:
        print(f'{_mode} seed {_seed}: existing metrics are invalid ({_exc}).', flush=True)
        return False
    _complete = _metrics.get('mode') == _mode and _metrics.get('run_seed') == _seed
    print(f'{_mode} seed {_seed}: valid result={_complete}.', flush=True)
    return _complete


_sweep_seed = 42
print(f'Starting sweep experiment: seed={_sweep_seed}, project={PROJECT_ROOT}', flush=True)
if _baseline_run_complete('sweep', _sweep_seed):
    print(f'Sweep seed {_sweep_seed}: đã có kết quả hợp lệ trên Drive, bỏ qua training.')
else:
    prepare_local_splits(only_subset=True)
    print(f'Sweep seed {_sweep_seed}: launching training subprocess...', flush=True)
    run_streaming(
        [sys.executable, '-u', '-m', 'experiments.baseline.experiment', '--mode', 'sweep', '--seed', str(_sweep_seed)],
        cwd=PROJECT_ROOT,
    )

/content
Starting sweep experiment: seed=42, project=/content/data-poisoning-plantvillage
Checking sweep seed 42: metrics=False, checkpoint=True
sweep seed 42: training is required.
Chép 23,904/23,904 ảnh từ Drive về /content/splits_local (đã có sẵn 0).
  đã chép 2,000/23,904 (117 ảnh/giây)
  đã chép 4,000/23,904 (101 ảnh/giây)
  đã chép 6,000/23,904 (116 ảnh/giây)
  đã chép 8,000/23,904 (117 ảnh/giây)
  đã chép 10,000/23,904 (118 ảnh/giây)
  đã chép 12,000/23,904 (120 ảnh/giây)
  đã chép 14,000/23,904 (120 ảnh/giây)
  đã chép 16,000/23,904 (118 ảnh/giây)
  đã chép 18,000/23,904 (117 ảnh/giây)
  đã chép 20,000/23,904 (116 ảnh/giây)
  đã chép 22,000/23,904 (117 ảnh/giây)
Xong 23,904 ảnh trong 204s.
data/splits đang trỏ tới /content/splits_local (đĩa local).
Sweep seed 42: launching training subprocess...
sweep experiments:   0%|          | 0/1 [00:00<?, ?experiment/s]
Starting experiment: mode=sweep, seed=42
Loading manifests: train=/content/splits_local/manifest_train.csv, test=/conten

In [11]:
%cd /content/data-poisoning-plantvillage
from experiments.baseline.experiment import load_config   # dùng lại hàm đọc config của experiment.py

_dataset_config, _ = load_config()
_confirm_seeds = _dataset_config['modes']['confirm']['seeds']
_missing_confirm_seeds = [
    _seed for _seed in _confirm_seeds
    if not _baseline_run_complete('confirm', _seed)
]
print(f'Confirm seeds đã có trên Drive: {[s for s in _confirm_seeds if s not in _missing_confirm_seeds]}', flush=True)
print(f'Confirm seeds cần chạy: {_missing_confirm_seeds}', flush=True)
if _missing_confirm_seeds:
    prepare_local_splits(only_subset=False)
for _i, _seed in enumerate(_missing_confirm_seeds, 1):
    print(f'\n=== Confirm seed {_seed} ({_i}/{len(_missing_confirm_seeds)}): launching training subprocess... ===', flush=True)
    run_streaming(
        [sys.executable, '-u', '-m', 'experiments.baseline.experiment', '--mode', 'confirm', '--seed', str(_seed)],
        cwd=PROJECT_ROOT,
    )
    print(f'Confirm seed {_seed}: subprocess completed.', flush=True)

print(f'Sweep: seed {_sweep_seed}; confirm: seeds {_confirm_seeds}.', flush=True)
print(f'Results are stored in: {DRIVE_RESULTS / "baseline"}')
print(f'Checkpoints are stored in: {DRIVE_CHECKPOINTS / "baseline"}')

/content
Checking confirm seed 42: metrics=False, checkpoint=False
confirm seed 42: training is required.
Checking confirm seed 43: metrics=False, checkpoint=False
confirm seed 43: training is required.
Checking confirm seed 44: metrics=False, checkpoint=False
confirm seed 44: training is required.
Confirm seeds đã có trên Drive: []
Confirm seeds cần chạy: [42, 43, 44]
Chép 30,401/54,305 ảnh từ Drive về /content/splits_local (đã có sẵn 23,904).
  đã chép 2,000/30,401 (104 ảnh/giây)
  đã chép 4,000/30,401 (105 ảnh/giây)
  đã chép 6,000/30,401 (107 ảnh/giây)
  đã chép 8,000/30,401 (107 ảnh/giây)
  đã chép 10,000/30,401 (106 ảnh/giây)
  đã chép 12,000/30,401 (107 ảnh/giây)
  đã chép 14,000/30,401 (108 ảnh/giây)
  đã chép 16,000/30,401 (110 ảnh/giây)
  đã chép 18,000/30,401 (109 ảnh/giây)
  đã chép 20,000/30,401 (108 ảnh/giây)
  đã chép 22,000/30,401 (109 ảnh/giây)
  đã chép 24,000/30,401 (110 ảnh/giây)
  đã chép 26,000/30,401 (111 ảnh/giây)
  đã chép 28,000/30,401 (111 ảnh/giây)
  đã chép

KeyboardInterrupt: 

In [12]:
# Xem kết quả baseline đã có trên Drive
import json
import pandas as pd

_rows = []
for _p in sorted((DRIVE_RESULTS / 'baseline').glob('*/seed_*/metrics.json')):
    _m = json.loads(_p.read_text(encoding='utf-8'))
    _rows.append({'mode': _m['mode'], 'seed': _m['run_seed'], 'train': _m['train_samples'],
                  'epochs': _m['epochs'], **_m['evaluation']})
if _rows:
    _df = pd.DataFrame(_rows)
    display(_df)
    display(_df.drop(columns=['seed', 'train', 'epochs']).groupby('mode').agg(['mean', 'std']))
else:
    print('Chưa có kết quả nào.')

,mode,seed,train,epochs,accuracy,precision,recall,f1_score
0,confirm,42,43429,5,0.946396,0.937608,0.936354,0.931388
1,sweep,42,13028,3,0.930857,0.914892,0.909783,0.905890


accuracy     precision        recall      f1_score    
             mean std      mean std      mean std      mean std
mode                                                           
confirm  0.946396 NaN  0.937608 NaN  0.936354 NaN  0.931388 NaN
sweep    0.930857 NaN  0.914892 NaN  0.909783 NaN  0.905890 NaN

In [13]:
# Chạy khi kết thúc phiên (sau khi đã xem xong kết quả). Sau lệnh này Drive bị ngắt:
# muốn chạy tiếp phải chạy lại cell mount (hoặc khởi động lại runtime).
finish_session()

Đã đồng bộ và ngắt kết nối Drive.
